# SportVision: annotate a sports clip

Developer preview. People and ball detection, tracking, jersey-color
team groups, and a possession estimate based on how close the ball is
to each player.

It does not produce validated match statistics and it does not calibrate
the field automatically. Track IDs are not player identities. A GPU
runtime is optional and makes it faster.


## Install


In [ ]:
!pip install -q "sportvision[app] @ git+https://github.com/MohibShaikh/sportvision"


## Upload a short clip

Ten seconds of a steady, wide view works best. Keep both teams and the
ball in shot.


In [ ]:
from google.colab import files

uploaded = files.upload()
VIDEO_PATH = next(iter(uploaded))
VIDEO_PATH


## Process up to 300 frames

The first run downloads the YOLO weights. `analyze_video` writes the
annotated MP4 and a JSON report next to it, and returns the same report.

An empty `possession` means there was not enough evidence to attribute
any frame. That is a real answer, not a failure.


In [ ]:
from sportvision.video import analyze_video

OUTPUT_PATH = "/content/analyzed.mp4"

# Jersey colours in BGR. Automatic grouping is the fallback and it
# needs the opening frames to decide, leaving them unassigned.
TEAM_COLORS = [(255, 255, 255), (21, 21, 21)]  # white kit, dark kit

stats = analyze_video(
    VIDEO_PATH, OUTPUT_PATH, max_frames=300, team_colors=TEAM_COLORS
)

print(f"frames reviewed:        {stats['frames_processed']}")
print(f"frames with a ball:     {stats['frames_with_tracked_ball']}")
print(f"frames used for poss.:  {stats['attributed_frames']}")
for team, share in stats["possession"].items():
    print(f"team {int(team) + 1} possession:     {share:.0%}")


## Preview a frame


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

cap = cv2.VideoCapture(OUTPUT_PATH)
ok, frame = cap.read()
cap.release()
if ok:
    cv2_imshow(frame)


## Tactical view

Project the players onto a top-down court. This needs the four corners
of the playing area, in pixels, read off the frame above in this order:
far left, far right, near right, near left.

The placeholder below uses the corners of the image itself, which is
almost certainly wrong for your clip. Replace the numbers with the real
court corners or the dots will land in the wrong places.

The projection assumes the camera holds still. After a pan or a zoom you
need new corners.


In [ ]:
from sportvision import radar

h, w = frame.shape[:2]
CORNERS = [[0, h * 0.55], [w, h * 0.55], [w, h], [0, h]]  # <- edit these
SPORT = "basketball"  # or "soccer", "tennis"

field = radar.calibrate(SPORT, CORNERS)
row = stats["review"]["frames"][0]
cv2_imshow(radar.draw_frame(SPORT, field, row, width_px=900))


## Download the results


In [ ]:
files.download(OUTPUT_PATH)
files.download("/content/analyzed.json")


## Run the review app locally

Colab cannot serve the interactive review UI, where you correct team
assignments, mark a missed ball, and click the court corners instead of
typing them. For that, run it on your own machine:

```bash
pip install "sportvision[app]"
sportvision app
```
